In [1]:
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
import util
import json
from pyfasttext import FastText
import pandas as pd
import pickle
import multiprocessing
from multiprocessing import Pool,Manager
from multiprocessing import Process, Manager

In [3]:
model_pubset_en = Doc2Vec.load("model/model_en/pubset_en")
model_pubset_de = Doc2Vec.load("model/model_de/pubSet_de_all.model")
model_language = FastText('model/lid.176.ftz')

In [2]:
with open("../../data/json/publication.json") as f :
    pubs = json.load(f)["_source"]
    
pubdf=pd.DataFrame(pubs).T


In [4]:
from Item import Item
item = Item()

def isDataset(item):
    if 'research_data'== getType(item):
        return True
    return False

def getType(item_id):
    dictItem = item.get(item_id)
    dictItem = dictItem._source
    return (dictItem["type"])

In [7]:
def getSimilarDataset(idf):
    words = util.get_Vocabs(item,idf)
    res_datast = []
    langg = model_language.predict_proba_single(" ".join(words))
    if (len(langg)==0):
        print(" ".join(words))
    else:
        lang = langg[0][0]
        #print("create infer_vector...")
        if lang=="en":
            paragvector = model_pubset_en.infer_vector(words)
            #paragraph_vectors.append(paragvector)
            result = model_pubset_en.docvecs.most_similar(positive=[paragvector], topn=50)
        else:
            paragvector = model_pubset_de.infer_vector(words)
            #paragraph_vectors.append(paragvector)
            result = model_pubset_de.docvecs.most_similar(positive=[paragvector], topn=50)
            
        #print("filter datasets...")
        res_datast = [(a,b) for (a,b) in result if isDataset(a)][0:10]  
    return res_datast

In [8]:
import multiprocessing as mp
import numpy as np
import json

#pool = mp.Pool(mp.cpu_count())
#similarity_all=[]

num_partitions = 20 #number of partitions to split dataframe
num_cores = mp.cpu_count() #number of cores on your machine
print(num_cores)
num_cores = 15

ids_all = pubdf.index.tolist()
#df_split = np.array_split(ids_all, num_cores, axis=0)

def func(idfs):
    with open('DocVec_similarity_matrix_top_10.txt', 'a') as b:
        b.write(json.dumps({"target_items":idfs,"similar_items":getSimilarDataset(idfs)}))
        b.write("\n")
        b.flush()

pool = Pool(num_cores)
pool.map(func, ids_all)
pool.close()
pool.join()



32


/home/tavakons/anaconda3/envs/py37/lib/python3.7/site-packages/gensim/matutils.py:737: FutureWarning: Conversion of the second argument of issubdtype from `int` to `np.signedinteger` is deprecated. In future, it will be treated as `np.int64 == np.dtype(int).type`.
  if np.issubdtype(vec.dtype, np.int):
/home/tavakons/anaconda3/envs/py37/lib/python3.7/site-packages/gensim/matutils.py:737: FutureWarning: Conversion of the second argument of issubdtype from `int` to `np.signedinteger` is deprecated. In future, it will be treated as `np.int64 == np.dtype(int).type`.
  if np.issubdtype(vec.dtype, np.int):
/home/tavakons/anaconda3/envs/py37/lib/python3.7/site-packages/gensim/matutils.py:737: FutureWarning: Conversion of the second argument of issubdtype from `int` to `np.signedinteger` is deprecated. In future, it will be treated as `np.int64 == np.dtype(int).type`.
  if np.issubdtype(vec.dtype, np.int):
/home/tavakons/anaconda3/envs/py37/lib/python3.7/site-packages/gensim/matutils.py:737: F

In [ ]:
#without pool and mutliprocessing
similarity_all=[]
for idf in pubdf.index:
    try:
        similarity_all.append({"target_items":idf,"similar_items":getSimilarDataset(idf)})
    except:
        print(idf)
    
with open('DocVec_similarity_matrix_top_10.pkl', 'wb') as b:
    pickle.dump(similarity_all,b)

In [30]:
df_split[0].tolist() 

['gesis-ssoar-1002',
 'gesis-ssoar-1006',
 'gesis-ssoar-10066',
 'gesis-ssoar-1008',
 'gesis-ssoar-10095',
 'gesis-ssoar-10096',
 'gesis-ssoar-10103',
 'gesis-ssoar-10111',
 'gesis-ssoar-1012',
 'gesis-ssoar-10124',
 'gesis-ssoar-10133',
 'gesis-ssoar-10139',
 'gesis-ssoar-10140',
 'gesis-ssoar-10149',
 'gesis-ssoar-1016',
 'gesis-ssoar-1017',
 'gesis-ssoar-10199',
 'gesis-ssoar-10235',
 'gesis-ssoar-10236',
 'gesis-ssoar-10241',
 'gesis-ssoar-10248',
 'gesis-ssoar-1025',
 'gesis-ssoar-10254',
 'gesis-ssoar-10256',
 'gesis-ssoar-10259',
 'gesis-ssoar-1026',
 'gesis-ssoar-10260',
 'gesis-ssoar-10273',
 'gesis-ssoar-10278',
 'gesis-ssoar-10284',
 'gesis-ssoar-10285',
 'gesis-ssoar-10287',
 'gesis-ssoar-10288',
 'gesis-ssoar-10289',
 'gesis-ssoar-1029',
 'gesis-ssoar-10302',
 'gesis-ssoar-10315',
 'gesis-ssoar-10318',
 'gesis-ssoar-10321',
 'gesis-ssoar-10327',
 'gesis-ssoar-10331',
 'gesis-ssoar-10338',
 'gesis-ssoar-10342',
 'gesis-ssoar-10343',
 'gesis-ssoar-10345',
 'gesis-ssoar-10347

In [11]:
#%time getSimilarDataset("gesis-ssoar-1006")
#%time getType("gesis-ssoar-1006")
#%time util.get_Vocabs(item,"gesis-ssoar-1006")

In [9]:
similarity_all = [line.rstrip('\n') for line in open("DocVec_similarity_matrix_top_10.txt")]

In [10]:
len(similarity_all)

96755

In [28]:
import ast

similarity_all = [line.rstrip('\n') for line in open("DocVec_similarity_matrix_top_10.txt")]

similarity_all_eval=[]

for sim in similarity_all:
    similarity_all_eval.append(ast.literal_eval(sim))
    
def getsimilarityjson(id):
    try:
        return [sim for sim in similarity_all_eval if sim['target_items']==id][0]
    except:
        return None

doc_id = "gesis-ssoar-10327"
sim = getsimilarityjson(doc_id)
print(sim)

{'target_items': 'gesis-ssoar-10327', 'similar_items': [['datasearch-httpwww-da-ra-deoaip--oaioai-da-ra-de657032', 0.3866425156593323], ['datasearch-httpwww-da-ra-deoaip--oaioai-da-ra-de461750', 0.3676287531852722], ['ZA4053', 0.3615070879459381], ['ZA5965', 0.34789931774139404], ['datasearch-httpwww-da-ra-deoaip--oaioai-da-ra-de467433', 0.3472675085067749], ['datasearch-httpwww-da-ra-deoaip--oaioai-da-ra-de4916', 0.3373015522956848], ['datasearch-httpwww-da-ra-deoaip--oaioai-da-ra-de663042', 0.3362707495689392], ['datasearch-httpwww-da-ra-deoaip--oaioai-da-ra-de5386', 0.3313177227973938], ['datasearch-httpwww-da-ra-deoaip--oaioai-da-ra-de7276', 0.3302221894264221], ['ZA1945', 0.3301737904548645]]}
